# 01 · Explore the UCI HAR inertial signals

This notebook opens a few accelerometer and gyroscope examples from UCI HAR. The aim is to understand the recorded windows before trying to model them. Each row is a short segment, not a complete recording.

## A. Sensors, axes, sample rate, and units

An accelerometer measures acceleration along X, Y, and Z. `total_acc` includes gravity and movement; `body_acc` is the estimated body-motion component after gravity was separated. A gyroscope measures angular velocity around three axes. UCI documents a sample rate of **50 Hz**, acceleration in units of gravity (**g**), and angular velocity in **radians/second (rad/s)**. Axis directions are sensor coordinates and depend on phone placement.

UCI has already filtered these signals and split them into overlapping 128-sample windows. The filters in the second notebook are additional experiments on the supplied signals.

## B. Load both splits and validate files

We load all nine inertial matrices per split: total acceleration, body acceleration, and gyroscope, each with three axes. Labels and participant IDs must align with the 128 samples in every row. The helper searches relative project locations and explains how to add the official dataset if it cannot find it.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
FS, N = 50, 128
KINDS, AXES = ("total_acc", "body_acc", "body_gyro"), ("x", "y", "z")
def project_root():
    for p in (Path.cwd(), *Path.cwd().parents):
        if (p/"notebooks"/"01_signal_exploration.ipynb").is_file(): return p
    raise FileNotFoundError("Open Jupyter from wearable-signal-lab/ or its notebooks/ folder.")
ROOT=project_root()
choices=[p/"data"/"raw"/"UCI HAR Dataset" for p in (ROOT,*ROOT.parents)]
DATA=next((p for p in choices if p.is_dir()),None)
if DATA is None:
    raise FileNotFoundError("UCI HAR data missing. Download from https://archive.ics.uci.edu/dataset/240/human+activity+recognition+using+smartphones, extract 'UCI HAR Dataset' to data/raw/ or reuse neighboring wearable-ai-101/data/raw/. See data/README.md. No fake data is made.")
a=pd.read_csv(DATA/"activity_labels.txt",sep=r"\s+",header=None,names=["id","name"])
ACT=dict(zip(a.id,a.name))
def load(split):
    d=DATA/split; sigdir=d/"Inertial Signals"
    paths=[d/f"y_{split}.txt",d/f"subject_{split}.txt"]+[sigdir/f"{kind}_{axis}_{split}.txt" for kind in KINDS for axis in AXES]
    missing=[str(p.relative_to(DATA)) for p in paths if not p.is_file()]
    if missing: raise FileNotFoundError(f"Missing {split} files: {missing}. Check data/README.md.")
    y=np.loadtxt(d/f"y_{split}.txt",dtype=int).reshape(-1); sub=np.loadtxt(d/f"subject_{split}.txt",dtype=int).reshape(-1)
    signals={}
    for kind in KINDS:
        signals[kind]={}
        for axis in AXES:
            p=sigdir/f"{kind}_{axis}_{split}.txt"
            v=pd.read_csv(p,sep=r"\s+",header=None).to_numpy(dtype=np.float32)
            if v.shape!=(len(y),N): raise ValueError(f"{p.name} has {v.shape}; expected ({len(y)}, {N}).")
            signals[kind][axis]=v
    if len(sub)!=len(y): raise ValueError(f"{split} subject IDs and labels are misaligned.")
    if set(y)-set(ACT): raise ValueError(f"Unknown activity IDs: {set(y)-set(ACT)}")
    return {"y":y,"subjects":sub,"signals":signals}
train,test=load("train"),load("test")
for name,data,expected in (("train",train,7352),("test",test,2947)):
    if len(data["y"])!=expected: raise ValueError(f"Expected {expected} {name} windows, got {len(data['y'])}.")
print(f"Data: {DATA}; train={len(train['y'])}, test={len(test['y'])}; each row={N} samples ({N/FS:.2f}s) at {FS} Hz")
print(f"NumPy {np.__version__} | pandas {pd.__version__} | Matplotlib {plt.matplotlib.__version__} | SciPy {scipy.__version__}")

## C. Inspect actual structure and labels

Confirm the matrices and activity names. Counts describe windows, not complete recordings.

In [ ]:
for split,data in (("TRAIN",train),("TEST",test)):
    print(f"\n{split}: {len(data['y'])} windows; {np.unique(data['subjects']).size} participants")
    for kind in KINDS: print(kind,{axis:data['signals'][kind][axis].shape for axis in AXES})
print("Activity labels:",ACT)
print("Documentation: 50 Hz; acceleration g; gyroscope rad/s.")

## D. Plot acceleration over time

We select the first WALKING row. Time zero is the beginning of this one 2.56-second segment. These curves do not represent a whole walk; UCI windows overlap by 50%. Each panel compares total and estimated body acceleration on one axis.

In [ ]:
idx=int(np.flatnonzero(train["y"]==1)[0]); activity=ACT[int(train["y"][idx])]
t=np.arange(N)/FS
fig,axs=plt.subplots(3,1,figsize=(10,7),sharex=True)
for ax,axis in zip(axs,AXES):
    ax.plot(t,train["signals"]["total_acc"][axis][idx],label="Total acceleration",alpha=.7)
    ax.plot(t,train["signals"]["body_acc"][axis][idx],label="Body acceleration")
    ax.set_ylabel(f"{axis.upper()} (g)");ax.grid(alpha=.25);ax.legend()
axs[0].set_title(f"One {activity} segment: total vs body acceleration");axs[-1].set_xlabel("Seconds in this window")
plt.tight_layout();plt.show()
print(f"Train row {idx}; {N/FS:.2f} s total; final sample timestamp {t[-1]:.2f} s.")

## E. Plot gyroscope over time

These axes show angular velocity in rad/s for the same window. Signal amplitude is the measured rotational speed; the sign gives direction in the sensor coordinate system.

In [ ]:
fig,axs=plt.subplots(3,1,figsize=(10,6),sharex=True)
for ax,axis in zip(axs,AXES):
    ax.plot(t,train["signals"]["body_gyro"][axis][idx],color="#7A5195")
    ax.set_ylabel(f"{axis.upper()} (rad/s)");ax.grid(alpha=.25)
axs[0].set_title(f"One {activity} segment: gyroscope");axs[-1].set_xlabel("Seconds in this window")
plt.tight_layout();plt.show()

## F. Visualize analysis windows

A window is a block of adjacent samples summarized together. We subdivide one supplied 128-sample row into 32-, 64-, and 128-sample pieces. This shows shorter local views; it does not create a longer recording. Durations are samples divided by 50 Hz.

In [ ]:
sig=train["signals"]["body_acc"]["x"][idx]
fig,axs=plt.subplots(3,1,figsize=(10,7),sharex=True)
for ax,size in zip(axs,(32,64,128)):
    ax.plot(t,sig)
    for edge in range(size,N,size):ax.axvline(edge/FS,color="red",ls="--",alpha=.65)
    ax.set_title(f"{size} samples = {size/FS:.2f} s per subwindow");ax.set_ylabel("Body accel X (g)");ax.grid(alpha=.2)
axs[-1].set_xlabel("Time within this original UCI row (s)");plt.tight_layout();plt.show()

## G. Glossary and exercises

- **Sampling frequency:** measurements per second; 50 Hz here.
- **Sample:** one measurement at one time on one axis.
- **Amplitude:** the signal value in its documented unit.
- **Axis:** a sensor coordinate, X/Y/Z; not automatically a body direction.
- **Window:** a finite block of consecutive samples.
- **Overlap:** samples shared between neighboring windows; UCI uses 50%.
- **Total acceleration:** gravity plus movement.
- **Body acceleration:** estimated movement after gravity separation.
- **Signal magnitude:** combined 3D signal size, without direction.

**Try independently:** (1) plot a SITTING row instead of WALKING; (2) compare total and body acceleration on all axes; (3) change subwindow sizes to 16, 32, and 128 samples; (4) compare the same activity from the test split. Explain why one row is not a continuous recording.